# Notebook 17 — is the extractor scoring correct transcriptions as wrong?

`transcription_correct` is an exact match between two canonicalized strings, so
the perception AUROC rests on that comparison being fair. This notebook opens
the comparison up and looks at it, image included.

For every item the scoring pipeline runs four stages, on the model samples and
on the ground truth alike:

| stage | function | produces |
|---|---|---|
| 1 | *(the run)* | raw model output |
| 2 | `pilot.parsing.parse_transcription` | the `**Answer:**` field |
| 3 | `pilot.canonicalize.extract_final_answer` | the final-answer span |
| 4 | `pilot.rescore.answer_label` | the string actually compared |

Offline inspection found four things worth seeing rather than taking on
trust, and this notebook is how you check them against the handwriting:

1. **A real bug.** `structural_clean` unwraps `\textcolor{}{}` with a `[^{}]*`
   body, so nested cases survive into the label. 85/300 ground truths, 59 of
   them `has_error=1` — FERMAT marks the *injected error* in red.
2. **A second real bug.** `extract_final_answer`'s last-line tier splits on
   `.` as a sentence terminator, so it also splits decimal numbers:
   `"the area is 75.46 cm."` extracts as `"46 cm"`.
3. **Cosmetic mismatches.** `2^3 = 8` vs `2^{3} = 8`; `0 = 9` vs `0 = 9,`;
   `11130 cm` vs `11130 \, cm`; `(x,z)` vs `(x, z)`.
4. **Scope mismatches.** The model reports `= 75.46 cm^2`; the ground truth
   spells out `= \pi r^2 = ... = 75.46 cm^2`. Same answer, scored wrong.

Bug 2 is why the bugs had to be *fixed* rather than noted: it truncated both
the ground truth and two model samples of item 101 to `"5 square meters"`, and
a looser rule then scored that shared mangling as agreement. A relaxation is
only trustworthy once the extractor feeding it is not mangling its inputs.

**No GPU and no model.** It reads the n=300 results CSV and the FERMAT images.
Runs in a couple of minutes.

**What this notebook does NOT do:** change the headline. `strict_v1` stays the
frozen rule of record — it produced every locked result and all of
`reference/*.json`. The looser rules are reported *alongside* it as a
sensitivity analysis. Moving a scoring rule after seeing that it raises
accuracy is the move this project keeps refusing to make.

In [1]:
# Auth + code access. No GPU/model needed -- this notebook only reads the
# dataset and an existing results CSV, it never runs generation.
import json
import os
import sys

from google.colab import drive
from huggingface_hub import login

drive.mount("/content/drive")
PROJECT_DIR = "/content/drive/MyDrive/uncertainty-math-vlm"
RESULTS_DIR = f"{PROJECT_DIR}/results"

# Reuses the token already cached on Drive by earlier notebooks.
with open(f"{PROJECT_DIR}/.tokens.json") as f:
    HF_TOKEN = json.load(f)["HF_TOKEN"]
login(token=HF_TOKEN)
print("Hugging Face login OK")

REPO_URL = "https://github.com/sepehrmaleki369/uncertainty-math-vlm.git"
!rm -rf repo
!git clone -q {REPO_URL} repo
%pip install -q -e repo/
# antlr4 pin: without it SymPy's LaTeX parser fails at CALL time, silently
# degrading every label to the plain-text tier. See
# pilot.canonicalize.latex_parser_available -- this cost 43/300 items once.
%pip install -q "antlr4-python3-runtime==4.11"

sys.path.insert(0, os.path.abspath("repo"))

# Purge any pilot.* left over from a previous clone in this runtime.
# importlib.invalidate_caches() does NOT reload already-imported modules, and
# a stale one produced a KeyError on the 2026-08-08 notebook-13 run for a
# symbol that was demonstrably on disk.
for _name in [m for m in sys.modules if m == "pilot" or m.startswith("pilot.")]:
    del sys.modules[_name]
import importlib
importlib.invalidate_caches()

import pilot.canonicalize
import pilot.data
import pilot.parsing
import pilot.rescore

print(f"pilot package imported from: {os.path.dirname(pilot.rescore.__file__)}")
assert pilot.canonicalize.latex_parser_available(), (
    "SymPy's LaTeX parser is NOT working. Every label falls back to plain text, "
    "which inflates entropy and deflates accuracy, and the numbers below will "
    "not match the offline analysis. Fix the antlr4 pin before continuing."
)
print("SymPy LaTeX parser: OK")

Mounted at /content/drive
Hugging Face login OK
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 15.2 MB/s eta 0:00:00
  Building editable for pilot (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
omegaconf 2.3.1 requires antlr4-python3-runtime==4.9.*, but you have antlr4-python3-runtime 4.11.0 which is incompatible.
pilot package imported from: /content/repo/pilot
SymPy LaTeX parser: OK


## 1. Load the run and rebuild the same sample

The CSV carries every raw model sample, so the whole scoring chain can be
re-run offline. The images are not in the CSV — they come from FERMAT, which
is gated, which is why this notebook exists in Colab rather than locally.

The `load_fermat_balanced` call reproduces the *same* draw the run used (same
`n`, `seed`, `target_error_frac`), and the assert below checks that row *i* of
the CSV really is item *i* of the sample rather than trusting the order.

In [2]:
import ast

import pandas as pd

RESULTS_CSV = "scaleup_n300_bal50_qwen25-vl-3b-instruct_20260802T163202Z.csv"
SEED, N_ITEMS, ERROR_FRAC = 42, 300, 0.5

df = pd.read_csv(f"{RESULTS_DIR}/{RESULTS_CSV}")
print(f"{len(df)} rows, model={df['model_id'].unique().tolist()}, "
      f"K={df['k_transcription'].unique().tolist()}")

sample = pilot.data.load_fermat_balanced(
    n=N_ITEMS, seed=SEED, target_error_frac=ERROR_FRAC)
assert len(sample) == len(df), f"{len(sample)} sample items vs {len(df)} CSV rows"

# load_fermat_balanced SHUFFLES its final selection, so index alignment is an
# assumption to verify, not one to make. Checking the question text pins the
# row-to-image mapping every display below depends on.
mismatched = [i for i in range(len(df))
              if sample[i]["orig_q"].strip() != str(df.iloc[i]["orig_q"]).strip()]
assert not mismatched, (
    f"{len(mismatched)} rows where the rebuilt sample's question does not match "
    f"the CSV's (first: {mismatched[:5]}). The images below would be attached to "
    "the wrong rows -- do not trust anything past this cell until resolved."
)
print("sample order matches the CSV on all 300 rows -- images are safely index-aligned")

300 rows, model=['Qwen/Qwen2.5-VL-3B-Instruct'], K=[5]


README.md:   0%|          | 0.00/3.74k [00:00<?, ?B/s]

data/train-00000-of-00010.parquet: reconstructing file:   0%|          |  0.00B /  467MB            

data/train-00000-of-00010.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00010.parquet: reconstructing file:   0%|          |  0.00B /  481MB            

...

data/train-00008-of-00010.parquet: downloading bytes:           |  0.00B            

data/train-00009-of-00010.parquet: reconstructing file:   0%|          |  0.00B /  493MB            

data/train-00009-of-00010.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2244 [00:00<?, ? examples/s]

sample order matches the CSV on all 300 rows -- images are safely index-aligned


## 2. The sensitivity table

Four cumulative rules (see `pilot/rescore.py` for the full definitions):

- **`strict_v1`** — exactly `canonical_answer_label`. Frozen; the rule of record.
- **`fixed_v2`** — v1 with both extractor bugs corrected (nested `\textcolor`,
  decimal splitting). *Bug fixes*, so they can move items either way, not only
  up — here 141 → 144, with individual items moving in both directions.
- **`relaxed_v3`** — v2 ignoring formatting the mathematics does not depend on.
- **`final_term_v4`** — v3 reduced to the term after the last top-level `=`.

v4 changes the **label**, not just the comparison, so entropy and correctness
stay derived from the same representation. Scoring correctness leniently while
leaving entropy strict would compare two different objects.

In [3]:
# ~2 min: this re-runs the full parse -> extract -> canonicalize chain for
# 300 items x 5 samples x 4 rules, with SymPy parsing on every label.
# progress=True because a silent multi-minute cell reads as a hang.
sens = pilot.rescore.scoring_sensitivity(df, n_boot=10000, seed=0, progress=True)

view = sens.copy()
view["accuracy"] = (view["accuracy"] * 100).round(1).astype(str) + "%"
view["AUROC [95% CI]"] = [f"{r.auroc:.3f} [{r.ci_low:.3f}, {r.ci_high:.3f}]"
                          for r in sens.itertuples()]
print(view[["rule", "n_correct", "accuracy", "AUROC [95% CI]",
            "excludes_chance", "n_at_max_entropy"]].to_string(index=False))

print(f"\naccuracy  {sens.accuracy.iloc[0]:.1%} -> {sens.accuracy.iloc[-1]:.1%}"
      f"   ({sens.n_correct.iloc[-1] - sens.n_correct.iloc[0]} more items scored correct)")
print(f"AUROC     {sens.auroc.iloc[0]:.3f} -> {sens.auroc.iloc[-1]:.3f}")
print("every rule excludes chance:", bool(sens.excludes_chance.all()))

[1/4] strict_v1: rescoring 300 items...


strict_v1:   0%|          | 0/300 [00:00<?, ?it/s]

          bootstrapping 10000 resamples...
[2/4] fixed_v2: rescoring 300 items...


fixed_v2:   0%|          | 0/300 [00:00<?, ?it/s]

          bootstrapping 10000 resamples...
[3/4] relaxed_v3: rescoring 300 items...


relaxed_v3:   0%|          | 0/300 [00:00<?, ?it/s]

          bootstrapping 10000 resamples...
[4/4] final_term_v4: rescoring 300 items...


final_term_v4:   0%|          | 0/300 [00:00<?, ?it/s]

          bootstrapping 10000 resamples...
         rule  n_correct accuracy       AUROC [95% CI]  excludes_chance  n_at_max_entropy
    strict_v1        141    47.0% 0.850 [0.806, 0.890]             True                50
     fixed_v2        144    48.0% 0.839 [0.794, 0.881]             True                50
   relaxed_v3        166    55.3% 0.798 [0.747, 0.846]             True                36
final_term_v4        190    63.3% 0.819 [0.769, 0.864]             True                24

accuracy  47.0% -> 63.3%   (49 more items scored correct)
AUROC     0.850 -> 0.819
every rule excludes chance: True


**Read it this way.** Accuracy moves a lot — the strict rule really is
undercounting correct reads. The AUROC decays gracefully and never touches
chance. A signal that existed only because of a pedantic string comparison
would not do that, so the perception result belongs to the entropy rather than
to the comparison.

`n_at_max_entropy` falls too: five samples that agree on the value but differ
in formatting stop looking like total disagreement.

## 3. Pick the items worth looking at

Four buckets, chosen by rule rather than by hand so the selection is
reproducible and cannot be tuned:

| bucket | definition | the question it answers |
|---|---|---|
| `cosmetic` | wrong at `strict_v1`, right at `relaxed_v3` | is the extractor too strict? |
| `scope` | wrong at `relaxed_v3`, right at `final_term_v4` | answer-only vs full chain |
| `tier_unstable` | ≥2 extractor branches across the 5 samples | is entropy measuring the extractor? |
| `genuinely_wrong` | wrong under **every** rule, entropy near max | what a real misread looks like |

In [4]:
import math

RULES = pilot.rescore.RULES
scored = {rule: pilot.rescore.rescore_run(df, rule) for rule in RULES}

n_tiers = [pilot.rescore.tier_instability(
    ast.literal_eval(r["all_transcription_samples_raw"]))["n_distinct_tiers"]
    for _, r in df.iterrows()]

flags = pd.DataFrame({
    "strict": scored["strict_v1"]["transcription_correct"].values,
    "fixed": scored["fixed_v2"]["transcription_correct"].values,
    "relaxed": scored["relaxed_v3"]["transcription_correct"].values,
    "final": scored["final_term_v4"]["transcription_correct"].values,
    "entropy": scored["strict_v1"]["perception_entropy"].values,
    "n_tiers": n_tiers,
    "has_error": df["has_error"].astype(bool).values,
})

MAX_H = math.log(int(df["k_transcription"].iloc[0]))
buckets = {
    "cosmetic":        flags.index[(~flags.strict) & flags.relaxed].tolist(),
    "scope":           flags.index[(~flags.relaxed) & flags.final].tolist(),
    "tier_unstable":   flags.index[(flags.n_tiers > 1) & (~flags.strict)].tolist(),
    "genuinely_wrong": flags.index[(~flags.final)
                                   & (flags.entropy > MAX_H - 1e-9)].tolist(),
}
for name, idx in buckets.items():
    print(f"{name:16s} {len(idx):3d} items   e.g. {idx[:8]}")

print(f"\nextractor used >1 branch across the 5 samples: "
      f"{int((flags.n_tiers > 1).sum())}/{len(flags)} items")
print(flags.groupby("n_tiers").agg(
    items=("entropy", "size"), mean_entropy=("entropy", "mean"),
    accuracy=("strict", "mean")).round(3).to_string())

cosmetic          27 items   e.g. [9, 34, 53, 64, 72, 73, 80, 87]
scope             26 items   e.g. [0, 1, 13, 20, 21, 31, 38, 49]
tier_unstable     95 items   e.g. [0, 8, 14, 17, 20, 21, 25, 26]
genuinely_wrong   29 items   e.g. [8, 17, 45, 47, 52, 60, 65, 81]

extractor used >1 branch across the 5 samples: 153/300 items
         items  mean_entropy  accuracy
n_tiers                               
1          147         0.685     0.565
2          128         1.031     0.383
3           25         1.243     0.360


That last table is the finding to sit with. Mean entropy rises monotonically
with the number of extractor branches used. Some of what the perception arm
scores as *model* uncertainty is the extractor changing its mind about which
line of an unchanged derivation is the answer.

**153/300 is a lower bound.** It counts only items where the extractor changed
*branch*. It misses the case where every sample used the same branch and the
extractor still picked a different block — item 9 below is exactly that: all
five samples hit `display_math`, three returned the conclusion `(x,z) \in R`
and two returned the intermediate step `x - z = (x-y)+(y-z)`. Entropy 1.332,
from five samples that agree mathematically.

## 4. The viewer

`show_item` prints the handwritten image and then the full four-stage trace for
all five samples and for the ground truth, under both the frozen rule and the
relaxed one, so you can see exactly where a verdict is decided.

Stage 4 is the only line that matters for the score — everything above it is
there to explain how stage 4 got its value.

In [5]:
import textwrap

import matplotlib.pyplot as plt

IMAGE_DIR = f"{PROJECT_DIR}/scoring_inspection_images"
os.makedirs(IMAGE_DIR, exist_ok=True)


def _wrap(label, value, width=100, indent=" " * 22):
    body = "None" if value is None else str(value).replace("\n", " ⏎ ")
    lines = textwrap.wrap(body, width) or [""]
    print(f"{label:<22}{lines[0]}")
    for line in lines[1:]:
        print(indent + line)


def show_item(i, rules=("strict_v1", "final_term_v4"), save=True, show_image=True):
    row = df.iloc[i]
    item = sample[i]
    samples_raw = ast.literal_eval(row["all_transcription_samples_raw"])

    print("=" * 108)
    print(f"ITEM {i}   has_error={bool(row['has_error'])}   "
          f"handwriting_style={row['handwriting_style']}   "
          f"image_quality={row['image_quality']}   "
          f"extractor branches used: {flags.loc[i, 'n_tiers']}")
    print(f"verdict by rule: " + "   ".join(
        f"{r}={bool(scored[r].loc[i, 'transcription_correct'])}" for r in RULES))
    print("=" * 108)
    _wrap("QUESTION:", row["orig_q"])

    if show_image:
        img = item["image"]
        if save:
            img.save(f"{IMAGE_DIR}/item{i:03d}.png")
        plt.figure(figsize=(9, 9 * img.height / max(img.width, 1)))
        plt.imshow(img)
        plt.axis("off")
        plt.title(f"item {i} — the handwritten page the model was shown")
        plt.show()

    for rule in rules:
        tr = pilot.rescore.trace_item(samples_raw, row["pert_a"], rule)
        print("-" * 108)
        print(f"RULE: {rule}    entropy={tr['perception_entropy']:.3f}    "
              f"majority {tr['majority_count']}/{len(samples_raw)}    "
              f"{'CORRECT' if tr['correct'] else 'WRONG'}")
        print("-" * 108)

        gt = tr["ground_truth"]
        print("GROUND TRUTH (pert_a)")
        _wrap("  [1] raw:", gt["raw"])
        _wrap("  [3] final answer:", gt["final_answer"])
        _wrap("  [4] LABEL:", gt["label"])
        print()

        for k, s in enumerate(tr["samples"]):
            mark = " <== matches GT" if s["label"] == gt["label"] else ""
            print(f"SAMPLE {k}   (extractor branch: {s['tier']})")
            _wrap("  [1] raw tail:", (s["raw"] or "")[-300:])
            _wrap("  [2] Answer field:", (s["answer_field"] or "")[:300])
            _wrap("  [3] final answer:", s["final_answer"])
            _wrap("  [4] LABEL:", str(s["label"]) + mark)
            print()
    print()


print("show_item(i) ready. Try one of the bucket indices printed above.")

show_item(i) ready. Try one of the bucket indices printed above.


### 4a. Cosmetic near-misses — the model read the page right

In [6]:
for i in buckets["cosmetic"][:3]:
    show_item(i)

ITEM 9   has_error=False   handwriting_style=True   image_quality=True   extractor branches used: 1
verdict by rule: strict_v1=False   fixed_v2=False   relaxed_v3=True   final_term_v4=True
QUESTION:             Let \( R \) be a relation from \( \mathbb{Q} \) to \( \mathbb{Q} \) defined by \( R = \{(a,b): a,b
                      \in \mathbb{Q} \text{ and } a - b \in \mathbb{Z}\} \). Show that ⏎ \begin{enumerate} ⏎     \item \(
                      (a,a) \in R \) for all \( a \in \mathbb{Q} \) ⏎     \item \( (a,b) \in R \) implies that \( (b,a)
                      \in R \) ⏎     \item \( (a,b) \in R \) and \( (b,c) \in R \) implies that \( (a,c) \in R \) ⏎
                      \end{enumerate} ⏎  ⏎
------------------------------------------------------------------------------------------------------------
RULE: strict_v1    entropy=1.055    majority 2/5    WRONG
------------------------------------------------------------------------------------------------------------
GROUND TRUTH 

### 4b. Scope mismatches — answer only vs the whole chain

In [7]:
for i in buckets["scope"][:2]:
    show_item(i)

ITEM 0   has_error=False   handwriting_style=True   image_quality=True   extractor branches used: 2
verdict by rule: strict_v1=False   fixed_v2=False   relaxed_v3=False   final_term_v4=True
QUESTION:             Find the number of 4 letter words, with or without meaning, which can be formed out of the letters
                      of the word ROSE, where the repetition of the letters is not allowed. ⏎  ⏎
------------------------------------------------------------------------------------------------------------
RULE: strict_v1    entropy=1.055    majority 2/5    WRONG
------------------------------------------------------------------------------------------------------------
GROUND TRUTH (pert_a)
  [1] raw:             There are as many words as there are ways of filling in 4 vacant places \_\_\_\_ by the 4 letters,
                      keeping in mind that the repetition is not allowed. The first place can be filled in 4 different
                      ways by anyone of the 4 letters

### 4c. Extractor instability — five samples, different branches

In [8]:
for i in buckets["tier_unstable"][:2]:
    show_item(i)

ITEM 0   has_error=False   handwriting_style=True   image_quality=True   extractor branches used: 2
verdict by rule: strict_v1=False   fixed_v2=False   relaxed_v3=False   final_term_v4=True
QUESTION:             Find the number of 4 letter words, with or without meaning, which can be formed out of the letters
                      of the word ROSE, where the repetition of the letters is not allowed. ⏎  ⏎
------------------------------------------------------------------------------------------------------------
RULE: strict_v1    entropy=1.055    majority 2/5    WRONG
------------------------------------------------------------------------------------------------------------
GROUND TRUTH (pert_a)
  [1] raw:             There are as many words as there are ways of filling in 4 vacant places \_\_\_\_ by the 4 letters,
                      keeping in mind that the repetition is not allowed. The first place can be filled in 4 different
                      ways by anyone of the 4 letters

### 4d. Genuinely wrong — what a real misread looks like

The control for everything above. If these look like the cosmetic cases, the
relaxation did not go far enough; if they look like real misreads, the buckets
are separating what they claim to.

In [9]:
for i in buckets["genuinely_wrong"][:2]:
    show_item(i)

ITEM 8   has_error=True   handwriting_style=True   image_quality=True   extractor branches used: 2
verdict by rule: strict_v1=False   fixed_v2=False   relaxed_v3=False   final_term_v4=False
QUESTION:             A and B together have Rs. 1210. If \(\frac{4}{15}\) of A's amount is equal to \(\frac{2}{5}\) of B's
                      amount, how much amount does B have? ⏎  ⏎
------------------------------------------------------------------------------------------------------------
RULE: strict_v1    entropy=1.609    majority 1/5    WRONG
------------------------------------------------------------------------------------------------------------
GROUND TRUTH (pert_a)
  [1] raw:             Option B ⏎  ⏎ \[ ⏎ \frac{4}{15} A = \frac{2}{5} B ⏎ \] ⏎  ⏎ \[ ⏎ \Rightarrow A = \left(\frac{2}{5}
                      \times \frac{15}{4}\right) B ⏎ \] ⏎  ⏎ \[ ⏎ \Rightarrow A = \textcolor{red}{\frac{8}{4}} B ⏎ \] ⏎  ⏎
                      \[ ⏎ \Rightarrow \frac{A}{B} = \frac{\textcolor{red}{8}}{4

## 5. Compact table of every disagreement

Every item where two rules disagree, with both labels side by side — for
scanning after the detailed reads above.

In [10]:
# For each flipped item, show the labels under the rule that FAILED it and
# under the rule that PASSED it. Showing only the strict labels would make a
# "cosmetic" item look like a genuine disagreement -- the whole point is that
# the two strict labels differ while the relaxed ones do not.
FAIL_PASS = {"cosmetic": ("strict_v1", "relaxed_v3"),
             "scope": ("relaxed_v3", "final_term_v4")}

rows = []
for bucket in ("cosmetic", "scope"):
    fail_rule, pass_rule = FAIL_PASS[bucket]
    for i in buckets[bucket]:
        rows.append({
            "i": i, "bucket": bucket,
            "H": round(float(flags.loc[i, "entropy"]), 3),
            f"{fail_rule} model": scored[fail_rule].loc[i, "majority_label"][:44],
            f"{fail_rule} truth": scored[fail_rule].loc[i, "gt_label"][:44],
            f"{pass_rule} (both)": scored[pass_rule].loc[i, "majority_label"][:44],
        })
disagree = pd.DataFrame(rows).sort_values(["bucket", "i"])
pd.set_option("display.max_colwidth", 48)
print(f"{len(disagree)} items where a looser rule changes the verdict")
print("  'cosmetic' rows: the two strict labels differ, the relaxed ones agree")
print("  'scope' rows   : relaxed still differs, matching only the final term fixes it\n")
for bucket in ("cosmetic", "scope"):
    sub = disagree[disagree.bucket == bucket].dropna(axis=1, how="all")
    print(f"--- {bucket} ({len(sub)}) " + "-" * 60)
    print(sub.drop(columns="bucket").to_string(index=False))
    print()

53 items where a looser rule changes the verdict
  'cosmetic' rows: the two strict labels differ, the relaxed ones agree
  'scope' rows   : relaxed still differs, matching only the final term fixes it

--- cosmetic (27) ------------------------------------------------------------
  i     H                              strict_v1 model                              strict_v1 truth                            relaxed_v3 (both)
  9 1.055                             text:(x,z) \in r                            text:(x, z) \in r                               text:(x,z)\inr
 34 0.500                                   text:0 = 9                                  text:0 = 9,                                     text:0=9
 53 1.332                 text:nc_{17} = 18c_{17} = 18 text:nc_{17} = \textcolor{red}{18c_{17}} = 1                     text:nc_{17}=18c_{17}=18
 64 1.332 text:\sin^{-1}(\sin\frac{3\pi}{5}) = \sin^{- text:\sin^{-1} ( \sin \frac{3\pi}{5} ) = \si text:\sin^{-1}(\sin\frac{3\pi}5)=\sin^{

## 6. What to carry out of this notebook

- **`strict_v1` remains the reported rule.** Every locked number and every
  `reference/*.json` snapshot uses it, and it is bit-identical to
  `canonical_answer_label` (locked by
  `pilot/tests/test_rescore.py::test_strict_v1_is_bit_identical_to_the_frozen_pipeline`).
- **The transcription accuracy we report is a floor, not an estimate.** Say so
  in the paper: the strict rule undercounts correct reads by ~16 points on this
  run. Quote it as a **range, 47–63%**, not as 63.3% — 9 of the 26 items
  `final_term_v4` newly scores correct rest on a ≤2-character match (`1`, `5`,
  `24`), where a wrong answer can land on the same token by coincidence. The
  other 17 are substantive.
- **The AUROC is robust to the scoring rule**, which is the claim the
  sensitivity table actually supports.
- **Both extractor defects are real and are now implemented correctly** —
  `canonicalize.unwrap_latex_macro` and `extract_final_answer(...,
  fix_decimal_split=True)`. The frozen entry points keep the old behaviour on
  purpose, each with a docstring saying why.
- **Relaxation is not automatically generous.** Item 101 was scored *correct*
  by a looser rule only because a bug had mangled both sides the same way.
  Fix the extractor before trusting any relaxed number.
- **Open, and worth a sentence in Limitations:** on 153/300 items the extractor
  fires different branches across the five samples, and mean entropy rises with
  that count. Part of the perception signal may be extractor instability rather
  than model uncertainty. Distinguishing them needs a rule where the extractor
  cannot vary — e.g. requiring `\boxed{}` in the prompt — which is a new run,
  not a rescoring.